# Filtracja bilateralna

## Konwolucja obrazu z filtrem o zadanych współczynnikach

Splot (konwolucję) obrazu wejściowego $I$ z filtrem $\psi$ dla ustalonego punktu obrazu $\mathbf{x}$ można przedstawić następująco:

\begin{equation}
\hat{I}(\mathbf{x}) = \frac{1}{W_N}\sum_{\mathbf{p} \in \eta(\mathbf{x})} \psi(||\mathbf{p}-\mathbf{x}||)I(\mathbf{p})
\end{equation}

gdzie:
- $\hat{I}$ jest obrazem wynikowym (przefiltrowanym),
- $W_N = \sum_y \psi(y)$ jest parametrem normalizującym współczynniki filtra $\psi$,
- $||\cdot||$ jest odległością między punktami obrazu $\mathbf{x}$ i $\mathbf{p}$ według ustalonej metryki (np. norma $L_2$). Uwaga, proszę pamiętać, że zarówno $\mathbf{x}$, jak i $\mathbf{p}$ to współrzędne przestrzenne,
- $\eta(\mathbf{x})$ jest otoczeniem punktu $\mathbf{x}$.

Funkcja $\psi$ decyduje o charakterze filtracji. Dla filtru Gaussowskiego:

\begin{equation}
\psi(y) = G_{\delta_s}(y)
\end{equation}

gdzie: $G_{\delta_s}(y)$ jest funkcją Gaussa z parametrem skali $\delta_s$.

Opisaną powyżej filtrację realizowaliśmy w ramach ćwiczenia "Przetwarzanie wstępne. Filtracja kontekstowa."

## Filtracja bilateralna

Wadą klasycznego splotu jest brak adaptacji współczynników filtra do lokalnego otoczenia $\eta(\mathbf{x})$ filtrowanego punktu $\mathbf{x}$.
Oznacza to wykorzystanie tych samych współczynników filtra $\psi$ niezależnie od tego czy otoczenie jest względnie jednorodne lub zawiera krawędzie obiektów (w tym przypadku dochodzi do "rozmywania" krawędzi).
Filtracja bilateralna uwzględnia lokalne otoczenie filtrowanego punktu, w ten sposób, że parametry filtra zmieniają się w zależności od "wyglądu" otocznia.


Współczynniki filtra obliczane są na podstawie odległości filtrowanego punktu $\mathbf{x}$ od każdego punktu otoczenia $\mathbf{p}$ w dziedzinie przestrzennej obrazu (tak jak przy typowym filtrze np. Gaussa) oraz odległości punktów w przeciwdziedzinie obrazu (np. na podstawie różnicy w jasności pikseli dla obrazu w odcieniach szarości):

\begin{equation}
\hat{I}(\mathbf{x}) = \frac{1}{W_N}\sum_{\mathbf{p} \in \eta(\mathbf{x})} \psi(||\mathbf{p}-\mathbf{x}||) \gamma(|I(\mathbf{p}) - I(\mathbf{x})|) I(\mathbf{p})
\end{equation}
gdzie:
- $W_N$ jest współczynnikiem normalizującym filtr,
- $\gamma$ jest funkcją odległości w przeciwdziedzinie obrazu, np. $\gamma(y)=\exp(-\frac{y^2}{2\delta_r^2})$
- parametr $\delta_r$ jest utożsamiany z poziomem szumu w obrazie i należy go dobrać w sposób empiryczny.

Proszę chwilę zastanowić się nad powyższym równaniem, w szczególności nad funkcją $\gamma$. Proszę wyznaczyć, jaka będzie wartość funkcji dla pikseli podobnych (różnica 0, 1, 2), a skrajnie różnych (255, 200).

##  Realizacja ćwiczenia

### Wczytanie danych

1. Wczytaj dane z pliku *MR_data.mat*. W tym celu wykorzystaj funkcję `loadmat` z pakietu scipy:
        from scipy.io import loadmat
        mat = loadmat('MR_data.mat')

2. Wczytany plik zawiera 5 obrazów: *I_noisefree*, *I_noisy1*, *I_noisy2*, *I_noisy3* oraz *I_noisy4*. Odczytać je można w następujący sposób:
        Input = mat['I_noisy1']

3.Wyświetl wybrany obraz z pliku *MR_data.mat*. Zagadka - co to za obrazowanie medyczne?

In [ ]:
import cv2
import os
import requests
from matplotlib import pyplot as plt
import numpy as np
from scipy import signal
from scipy.io import loadmat
import math

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/07_Bilateral/'

fileNames = ["MR_data.mat"]
for fileName in fileNames:
  if not os.path.exists(fileName):
      r = requests.get(url + fileName, allow_redirects=True)
      open(fileName, 'wb').write(r.content)

#TODO Samodzielna



In [ ]:
mat = loadmat('MR_data.mat')
Input_table= ['I_noisefree', 'I_noisy1', 'I_noisy2', 'I_noisy3', 'I_noisy4']
plt.figure(figsize=(20,20))
for i in range(len(Input_table)):
  plt.subplot(1, len(Input_table), i+1)
  plt.title(Input_table[i])
  plt.imshow(mat[Input_table[i]], cmap='gray')
plt.show()

### "Klasyczna" konwolucja

1. Zdefiniuj parametry filtra Gaussowskiego: rozmiar okna i wariancję $\delta_S$.
2. Oblicz współczynniki filtra na podstawie zdefiniowanych parametrów (najprościej w ramach podwójnej pętli for).
2. Sprawdź ich poprawność i zwizualizuj filtr (tak jak w ćwiczeniu pt. "Przetwarzanie wstępne. Filtracja kontekstowa.").
3. Wykonaj kopię obrazu wejściowego: `IConv = Input.copy()`
4. Wykonaj podwójną pętlę po obrazie. Pomiń ramkę, dla której nie jest zdefiniowany kontekst o wybranej wielkości.
5. W każdej iteracji stwórz dwuwymiarową tablicę zawierającą aktualny kontekst.
6. Napisz funkcję, która będzie obliczała nową wartość piksela.
Argumentem tej funkcji są aktualnie przetwarzane okno i współczynniki filtra.
7. Obliczoną wartość przypisz do odpowiedniego piksela kopii obrazu wejściowego.
8. Wyświetl wynik filtracji.
9. Porównaj wynik z obrazem oryginalnym.

In [ ]:
#TODO Samodzielna
def mesh(fun, size):
    fig = plt.figure()
    ax = fig.add_subplot(projection = '3d')


    X = np.arange(-size//2, size//2, 1)
    Y = np.arange(-size//2, size//2, 1)
    X, Y = np.meshgrid(X, Y)
    Z = fun

    ax.plot_surface(X, Y, Z)

    plt.show()
def classic_convolution(img, win_size, w_s):
  kernel1=signal.windows.gaussian(win_size, w_s)
  kernel2=np.outer(kernel1, kernel1) #filtr 2d
  kernel2=kernel2/np.sum(kernel2)
  Iconv=img.copy().astype(float)
  row,col=img.shape
  padding=win_size//2
  for i in range(padding, row-padding):
    for j in range(padding, col-padding):
      sasiedzi=img[i-padding:i+padding+1, j-padding:j+padding+1] #pobieramy otoczenie
      Iconv[i,j]=np.sum(sasiedzi * kernel2) #srednia wazozna (srodek najwieksza waga)
  Iconv=np.clip(Iconv, 0, 255)
  return kernel2, Iconv.astype(np.uint8)

for name in Input_table:
    mask, Iconv = classic_convolution(mat[name], 5, 1)
    plt.figure(figsize=(10,10))
    plt.subplot(1,2,1)
    plt.title(name)
    plt.imshow(mat[name], cmap='gray')
    plt.subplot(1,2,2)
    plt.title("classic convolution")
    plt.imshow(Iconv, cmap='gray')
    mesh(mask, 5)
    plt.show()
#obraz wygladzony i usuniety suzm, ale rozmywa krawedzie

### Filtracja bilateralna

1. Zdefiniuj dodatkowy parametr: wariancję $\delta_R$.
3. Wykonaj kopię obrazu wejściowego: `IBilateral = Input.copy()`
4. Wykonaj podwójną pętlę po obrazie. Pomiń ramkę, dla której nie jest zdefiniowany kontekst o wybranej wielkości.
5. W każdej iteracji stwórz dwuwymiarową tablicę zawierającą aktualny kontekst.
6. Napisz funkcję, która będzie obliczała nową wartość piksela.
Argumentami funkcji są aktualnie przetwarzane okno, współczynniki filtra gausowskiego (takie same jak wcześniej) i wariancja $\delta_R$.
7. Oblicz odległość w przeciwdziedzinie (dla wartości pikseli).
8. Oblicz funkcję Gaussa dla obliczonych odległości z zadanym parametrem.
9. Wykonaj normalizację obliczonych współczynników.
10. Obliczoną wartość przypisz do odpowiedniego piksela kopii obrazu wejściowego.
11. Wyświetl wynik filtracji.
12. Porównaj wynik z obrazem oryginalnym.

In [ ]:
def nowy_piksel(sasiedzi, kernel, w_r):
  center= sasiedzi[sasiedzi.shape[0]//2, sasiedzi.shape[1]//2]
  roznica=sasiedzi-center
  gamma=np.exp(-roznica**2/(2*w_r**2))
  kernel_bilateral= kernel * gamma
  kernel_bilateral_norm= np.sum(kernel_bilateral* sasiedzi) /np.sum(kernel_bilateral)
  return kernel_bilateral_norm
def bilateral_filter(img, win_size, w_s, w_r):
  kernel1=signal.windows.gaussian(win_size, w_s)
  kernel2=np.outer(kernel1, kernel1)
  padding=win_size//2
  row,col=img.shape
  ImgBilateral=img.copy().astype(float)
  for i in range(padding, row-padding):
    for j in range(padding, col-padding):
      sasiedzi=img[i-padding:i+padding+1, j-padding:j+padding+1]
      ImgBilateral[i,j]=nowy_piksel(sasiedzi, kernel2, w_r)
  ImgBilateral= ImgBilateral.astype(np.uint8)
  return ImgBilateral.astype(np.uint8)

for name in Input_table:
    IBilateral_low = bilateral_filter(mat[name], 5, 1, 10)
    IBilateral_high = bilateral_filter(mat[name], 5, 1, 50)

    plt.figure(figsize=(15,15))
    plt.subplot(1,3,1)
    plt.imshow(mat[name], cmap='gray')
    plt.title(name + " original")
    plt.subplot(1,3,2)
    plt.imshow(IBilateral_low, cmap='gray')
    plt.title("bilateral low w_r")
    plt.subplot(1,3,3)
    plt.imshow(IBilateral_high, cmap='gray')
    plt.title("bilateral high w_r")
    plt.show()
#bilaterusuwa szum ale zachowuje krawedzie
#duze w_r mniej zachowuje krawdzie